# 01 · Search & discovery stress test

Keyword-driven exploration using **ulykke, drukning, flom, skred**, plus the
first live exercise of `paths()` / `tables()` against SSB's real ~3800-table
index (previously only unit-tested against a fake API).

Findings roll up into [`TESTING_AND_IMPROVEMENTS.md`](../../TESTING_AND_IMPROVEMENTS.md).

In [1]:
import warnings

import pandas as pd

from ssb_statistikkbanken import SSB

pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 80)

ssb = SSB()
ssb

SSB(language='no', timeout=30, max_workers=4, cache=True)

## 1 · The four keywords

`search` is a free-text query across titles, variable names **and value
labels** — which turns out to matter a lot below.

In [2]:
hits = {}
for keyword in ["ulykke", "drukning", "flom", "skred"]:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        hits[keyword] = ssb.search(keyword, limit=100)
    print(f"{keyword:<10} {len(hits[keyword]):>3} hits")

ulykke      21 hits
drukning     0 hits
flom         5 hits


skred        1 hits


### `drukning` → 0 hits

An empty search returns a **correctly-shaped empty DataFrame**, not `None` and
not an exception — so `len(...)`, `.empty` and column access all still work.
That is the behaviour you want; worth confirming rather than assuming.

In [3]:
drukning = hits["drukning"]
print(f"empty: {drukning.empty} | rows: {len(drukning)}")
print(f"columns preserved: {list(drukning.columns)}")

empty: True | rows: 0
columns preserved: ['id', 'label', 'description', 'updated', 'firstPeriod', 'lastPeriod', 'timeUnit', 'variables', 'path', 'source']


### `skred` → 1 hit, and it is a **false positive**

The only match is *"Etternavn brukt av 200 personer eller flere"* (surnames used
by 200+ people). Nothing to do with landslides — it matches because SSB's search
looks inside **value labels**, and some Norwegian surname contains "skred".

Not a bug, but a real usability trap: a keyword search can return a table whose
*title* has nothing to do with the query.

In [4]:
hits["skred"][["id", "label", "timeUnit", "firstPeriod", "lastPeriod"]]

,id,label,timeUnit,firstPeriod,lastPeriod
0,12891,12891: Etternavn brukt av 200 personer eller flere 2018-2025,Annual,2018,2025


### `ulykke` → 21 hits, including **monthly** tables

This is the useful one. Note `timeUnit` — several are `Monthly`, which is what
notebook 03 uses to exercise the `date` derivation path.

In [5]:
ulykke = hits["ulykke"]
print(ulykke["timeUnit"].value_counts().to_string())
ulykke[["id", "label", "timeUnit", "firstPeriod", "lastPeriod"]].head(10)

timeUnit
Annual     17
Monthly     4


,id,label,timeUnit,firstPeriod,lastPeriod
0,03610,03610: Drepte eller skadde i trafikkulykker med vogntog innblandet 2003M01-2...,Monthly,2003M01,2026M06
1,11343,"11343: Rapporterte arbeidsulykker, etter næring (SN2007) og type ulykke 2015...",Annual,2015,2024
2,09006,"09006: Drepte eller skadde i trafikkulykker, etter politidistrikt (F) 1999-2025",Annual,1999,2025
3,05247,"05247: Kjøretøy i trafikkulykker med personskade, etter kjøretøytype, type d...",Monthly,2004M01,2026M06
4,09000,"09000: Drepte eller skadde i trafikkulykker, etter alder, kjønn, skadegrad, ...",Annual,1999,2025
5,08329,"08329: Drepte eller skadde i trafikkulykker, etter alder, kjønn, skadegrad, ...",Monthly,1999M01,2026M06
6,10141,"10141: Konsultasjoner hos fastlegen, etter alder, kjønn og diagnose 2012-2025",Annual,2012,2025
7,10903,"10903: Legevaktkonsultasjoner, etter alder, kjønn og diagnose 2014-2025",Annual,2014,2025
8,12044,"12044: Personer drept eller skadd i veitrafikkulykker, etter politidistrikt ...",Annual,1999,2025
9,10209,"10209: Drepte eller skadde i trafikkulykker, etter politidistrikt (K) 1999M0...",Monthly,1999M01,2026M06


In [6]:
# The naturfare (natural hazard) tables behind the `flom` hits
hits["flom"][["id", "label", "timeUnit", "firstPeriod", "lastPeriod"]]

,id,label,timeUnit,firstPeriod,lastPeriod
0,14347,"14347: Veger, etter vegtype innen aktsomhetsområder for naturfare (F) 2024-2026",Annual,2024,2026
1,14346,"14346: Bygninger, etter bygningstype innen aktsomhetsområder for naturfare (...",Annual,2024,2026
2,13902,13902: Bygninger og veger innen kartlagte naturfarearealer (K) 2024-2026,Annual,2024,2026
3,14345,14345: Bygninger og veger innen aktsomhetsområder for naturfare (K) 2024-2026,Annual,2024,2026
4,13008,13008: Antall planer med innsigelser og antall innsigelsesbegrunnelser til k...,Annual,2015,2025


## 2 · `paths()` and `tables()` — first live run

Both download SSB's full table index. The docstrings warn this is expensive;
measure whether that warning is proportionate.

In [7]:
import time

t0 = time.time()
paths = ssb.paths()
first = time.time() - t0

t0 = time.time()
tables_al = ssb.tables("al")
second = time.time() - t0

print(f"paths():      {len(paths):>5} paths   {first:.2f}s  (downloads the full index)")
print(f"tables('al'): {len(tables_al):>5} tables  {second:.2f}s  (same index, now cached)")
paths.head()

paths():       1202 paths   1.01s  (downloads the full index)
tables('al'):   374 tables  0.02s  (same index, now cached)


,id,label
0,FRA_GAMMEL_00450,Avslutta tidsserier
1,SBMENU1001,Levevaner og sosial kontakt
2,SBMENU1002,"Sykelighet, symptomer og medisinbruk"
3,SBMENU10022,"Lengre tidsserier for straffereaksjoner, utvalgte reaksjoner"
4,SBMENU10023,Straffede personer


The full-index download is ~1–2s once, then effectively free for the rest of the
session because pxwebpy caches the response. The docstring warning is fair but
the cost is lower than it implies.

## 3 · Picking the tables the rest of the suite uses

| Table | Why it was picked |
|---|---|
| `08329` | Monthly traffic-accident table — 330 periods, exercises `date` derivation and batching |
| `13902` / `14345` | Naturfare (flom/skred) — mix counts and km under one ContentsCode |
| `14346` / `14347` | The same data split into single-unit tables — the control group |
| `11343` | Work accidents by **næring (SN2007)** — links to KLASS `NARING_SN` |

In [8]:
for table_id in ["08329", "13902", "14346", "14347", "11343"]:
    info = ssb.describe(table_id)
    contents = list(info["ContentsCode"].values.values())
    print(f"{table_id}  time={info.time_unit:<10} decimals={info.decimals}  contents={contents}")

08329  time=monthly    decimals=0  contents=['Personer drept eller skadd']
13902  time=annual     decimals=0  contents=['Bygninger', 'Veger (km)']
14346  time=annual     decimals=0  contents=['Bygninger']
14347  time=annual     decimals=1  contents=['Veger (km)']


11343  time=annual     decimals=0  contents=['Rapporterte arbeidsulykker', 'Rapporterte arbeidsulykker (per 1000 ansatte)']
